
# Tratamento de Dados — Dashboard Comercial/Vendas

Este notebook foi criado para demonstrar um fluxo de tratamento de dados.

Base utilizada:
- `vendas_raw`
- `produtos_raw`
- `clientes_raw`
- `metas`
- `dicionario_dados`

Objetivo:
1. Ler bases brutas em Excel;
2. Identificar problemas de qualidade;
3. Tratar nulos, duplicidades, inconsistências e outliers;
4. Criar colunas calculadas;
5. Exportar uma base tratada para uso no Power BI.


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

arquivo = Path("base_comercial_raw.xlsx")

if not arquivo.exists():
    arquivo = Path("base_comercial_raw.xlsx")

vendas = pd.read_excel(arquivo, sheet_name="vendas_raw")
produtos = pd.read_excel(arquivo, sheet_name="produtos_raw")
clientes = pd.read_excel(arquivo, sheet_name="clientes_raw")
metas = pd.read_excel(arquivo, sheet_name="metas")

vendas.head()


,id_pedido,data_pedido,id_cliente,id_produto,vendedor,canal,quantidade,preco_unitario,desconto,frete,status_pedido
0,PED00001,2025-08-26 00:00:00,CLI115,PRD021,NaN,Inside Sales,4,908.99,181.80,6.27,dev
1,PED00002,2025-06-14 00:00:00,CLI157,PRD004,Bruno Costa,Market Place,2,"1,013.19",0.00,18.42,Entregue
2,PED00003,2025-12-13 00:00:00,CLI114,PRD017,Gustavo Reis,Market Place,2,"1,265.85",126.58,13.82,Cancelado
3,PED00004,2025-10-26 00:00:00,CLI022,PRD024,Bruno Costa,Marketplace,2,50.68,15.20,54.84,ENTREGUE
4,PED00005,2025-03-07 00:00:00,CLI151,PRD023,Diego Rocha,Loja Física,1,256.85,12.84,32.86,ENTREGUE



## 1. Diagnóstico inicial da qualidade dos dados


In [2]:

def diagnostico(df, nome):
    print(f"\n===== {nome.upper()} =====")
    print(f"Linhas: {df.shape[0]}")
    print(f"Colunas: {df.shape[1]}")
    print("\nTipos:")
    display(df.dtypes.to_frame("tipo"))
    print("\nNulos por coluna:")
    display(df.isna().sum().to_frame("qtd_nulos"))
    print("\nDuplicados totais:")
    print(df.duplicated().sum())

diagnostico(vendas, "vendas")
diagnostico(produtos, "produtos")
diagnostico(clientes, "clientes")
diagnostico(metas, "metas")



===== VENDAS =====
Linhas: 1300
Colunas: 11

Tipos:


,tipo
id_pedido,str
data_pedido,object
id_cliente,str
id_produto,str
vendedor,str
canal,str
quantidade,int64
preco_unitario,float64
desconto,float64
frete,float64



Nulos por coluna:


,qtd_nulos
id_pedido,0
data_pedido,3
id_cliente,2
id_produto,0
vendedor,173
canal,106
quantidade,0
preco_unitario,0
desconto,0
frete,0



Duplicados totais:
0

===== PRODUTOS =====
Linhas: 28
Colunas: 7

Tipos:


...


Duplicados totais:
0

===== METAS =====
Linhas: 72
Colunas: 4

Tipos:


,tipo
mes,datetime64[us]
vendedor,str
meta_receita,int64
meta_pedidos,int64



Nulos por coluna:


,qtd_nulos
mes,0
vendedor,0
meta_receita,0
meta_pedidos,0



Duplicados totais:
0



## 2. Cópia das bases brutas


In [3]:

vendas_trat = vendas.copy()
produtos_trat = produtos.copy()
clientes_trat = clientes.copy()
metas_trat = metas.copy()



## 3. Tratamento de datas



In [4]:

vendas_trat["data_pedido"] = pd.to_datetime(
    vendas_trat["data_pedido"],
    errors="coerce",
    dayfirst=True
)

clientes_trat["data_cadastro"] = pd.to_datetime(
    clientes_trat["data_cadastro"],
    errors="coerce",
    dayfirst=True
)

metas_trat["mes"] = pd.to_datetime(
    metas_trat["mes"],
    errors="coerce"
)

# Remover datas placeholder
vendas_trat.loc[vendas_trat["data_pedido"] <= "1900-01-01", "data_pedido"] = pd.NaT
clientes_trat.loc[clientes_trat["data_cadastro"] <= "1900-01-01", "data_cadastro"] = pd.NaT

print("Datas inválidas/nulas em vendas:", vendas_trat["data_pedido"].isna().sum())
print("Datas inválidas/nulas em clientes:", clientes_trat["data_cadastro"].isna().sum())


Datas inválidas/nulas em vendas: 5
Datas inválidas/nulas em clientes: 3



## 4. Padronização de categorias

Aqui tratamos inconsistências comuns de digitação e variação de escrita.


In [5]:

# Padronizar canal de venda
map_canal = {
    "Ecommerce": "E-commerce",
    "e-commerce": "E-commerce",
    "Loja Fisica": "Loja Física",
    "Market Place": "Marketplace",
    "Inside sales": "Inside Sales"
}

vendas_trat["canal"] = (
    vendas_trat["canal"]
    .replace(map_canal)
    .fillna("Não informado")
)

# Padronizar status do pedido
map_status = {
    "entregue": "Entregue",
    "ENTREGUE": "Entregue",
    "cancelado": "Cancelado",
    "dev": "Devolvido"
}

vendas_trat["status_pedido"] = (
    vendas_trat["status_pedido"]
    .replace(map_status)
    .fillna("Não informado")
)

# Padronizar regiões
map_regiao = {
    "sudeste": "Sudeste",
    "SUDSTE": "Sudeste",
    "nordeste": "Nordeste",
    "Centro Oeste": "Centro-Oeste"
}

clientes_trat["regiao"] = (
    clientes_trat["regiao"]
    .replace(map_regiao)
    .fillna("Não informado")
)

# Padronizar segmento
map_segmento = {
    "PF": "Pessoa Física",
    "PJ": "Pessoa Jurídica"
}

clientes_trat["segmento"] = (
    clientes_trat["segmento"]
    .replace(map_segmento)
    .fillna("Não informado")
)

# Nulos em vendedor
vendas_trat["vendedor"] = vendas_trat["vendedor"].fillna("Não informado")

display(vendas_trat[["canal", "status_pedido", "vendedor"]].head())
display(clientes_trat[["regiao", "segmento"]].head())


,canal,status_pedido,vendedor
0,Inside Sales,Devolvido,Não informado
1,Marketplace,Entregue,Bruno Costa
2,Marketplace,Cancelado,Gustavo Reis
3,Marketplace,Entregue,Bruno Costa
4,Loja Física,Entregue,Diego Rocha


,regiao,segmento
0,Nordeste,Pessoa Jurídica
1,Sul,Não informado
2,Norte,Pessoa Jurídica
3,Sudeste,Pessoa Física
4,Nordeste,Pessoa Jurídica



## 5. Tratamento de duplicidades

Duplicidades são tratadas conforme o campo-chave de cada tabela.


In [6]:

print("Pedidos duplicados antes:", vendas_trat.duplicated(subset=["id_pedido"]).sum())
print("Produtos duplicados antes:", produtos_trat.duplicated(subset=["id_produto"]).sum())
print("Clientes duplicados antes:", clientes_trat.duplicated(subset=["id_cliente"]).sum())

# Mantém a primeira ocorrência
vendas_trat = vendas_trat.drop_duplicates(subset=["id_pedido"], keep="first")
produtos_trat = produtos_trat.drop_duplicates(subset=["id_produto"], keep="first")
clientes_trat = clientes_trat.drop_duplicates(subset=["id_cliente"], keep="first")

print("Pedidos duplicados depois:", vendas_trat.duplicated(subset=["id_pedido"]).sum())
print("Produtos duplicados depois:", produtos_trat.duplicated(subset=["id_produto"]).sum())
print("Clientes duplicados depois:", clientes_trat.duplicated(subset=["id_cliente"]).sum())


Pedidos duplicados antes: 1
Produtos duplicados antes: 0
Clientes duplicados antes: 1
Pedidos duplicados depois: 0
Produtos duplicados depois: 0
Clientes duplicados depois: 0



## 6. Validação de valores numéricos

Aqui tratamos:
- quantidade negativa;
- preços nulos;
- custos nulos;
- descontos maiores que o valor bruto;
- outliers de preço.


In [7]:

vendas_trat.loc[vendas_trat["quantidade"] < 0, "quantidade"] = np.nan


vendas_trat["quantidade"] = vendas_trat["quantidade"].fillna(1)


produtos_trat["custo_unitario"] = pd.to_numeric(produtos_trat["custo_unitario"], errors="coerce")
produtos_trat["preco_tabela"] = pd.to_numeric(produtos_trat["preco_tabela"], errors="coerce")

produtos_trat["custo_unitario"] = produtos_trat.groupby("categoria")["custo_unitario"].transform(
    lambda s: s.fillna(s.median())
)

produtos_trat["preco_tabela"] = produtos_trat.groupby("categoria")["preco_tabela"].transform(
    lambda s: s.fillna(s.median())
)


q1 = vendas_trat["preco_unitario"].quantile(0.25)
q3 = vendas_trat["preco_unitario"].quantile(0.75)
iqr = q3 - q1

limite_superior = q3 + 1.5 * iqr

vendas_trat["flag_outlier_preco"] = vendas_trat["preco_unitario"] > limite_superior

mediana_preco = vendas_trat.loc[~vendas_trat["flag_outlier_preco"], "preco_unitario"].median()
vendas_trat.loc[vendas_trat["flag_outlier_preco"], "preco_unitario"] = mediana_preco


vendas_trat["valor_bruto"] = vendas_trat["quantidade"] * vendas_trat["preco_unitario"]
vendas_trat.loc[vendas_trat["desconto"] > vendas_trat["valor_bruto"], "desconto"] = 0

display(vendas_trat[["quantidade", "preco_unitario", "desconto", "valor_bruto", "flag_outlier_preco"]].head())


,quantidade,preco_unitario,desconto,valor_bruto,flag_outlier_preco
0,4.00,908.99,181.80,"3,635.96",False
1,2.00,"1,013.19",0.00,"2,026.38",False
2,2.00,"1,265.85",126.58,"2,531.70",False
3,2.00,50.68,15.20,101.36,False
4,1.00,256.85,12.84,256.85,False



## 7. Validação de chaves entre fato e dimensões

Vamos verificar pedidos com produtos ou clientes inexistentes nas tabelas dimensão.


In [8]:

ids_produtos_validos = set(produtos_trat["id_produto"])
ids_clientes_validos = set(clientes_trat["id_cliente"])

vendas_trat["flag_produto_inexistente"] = ~vendas_trat["id_produto"].isin(ids_produtos_validos)
vendas_trat["flag_cliente_inexistente"] = ~vendas_trat["id_cliente"].isin(ids_clientes_validos)

print("Pedidos com produto inexistente:", vendas_trat["flag_produto_inexistente"].sum())
print("Pedidos com cliente inexistente:", vendas_trat["flag_cliente_inexistente"].sum())


vendas_trat = vendas_trat[
    (~vendas_trat["flag_produto_inexistente"]) &
    (~vendas_trat["flag_cliente_inexistente"]) &
    (vendas_trat["data_pedido"].notna())
].copy()

vendas_trat.shape


Pedidos com produto inexistente: 2
Pedidos com cliente inexistente: 2


(1290, 15)


## 8. Criação de colunas calculadas para o Power BI

Estas colunas ajudam a construir KPIs comerciais.


In [9]:

# Valor líquido
vendas_trat["valor_liquido"] = (
    vendas_trat["valor_bruto"] 
    - vendas_trat["desconto"] 
    + vendas_trat["frete"]
)

# Mês para relacionamento com metas
vendas_trat["mes"] = vendas_trat["data_pedido"].values.astype("datetime64[M]")

# Enriquecimento com produto para cálculo de custo e margem
vendas_trat = vendas_trat.merge(
    produtos_trat[["id_produto", "produto", "categoria", "subcategoria", "custo_unitario"]],
    on="id_produto",
    how="left"
)

vendas_trat["custo_total"] = vendas_trat["quantidade"] * vendas_trat["custo_unitario"]
vendas_trat["margem_bruta"] = vendas_trat["valor_liquido"] - vendas_trat["custo_total"]
vendas_trat["margem_%"] = np.where(
    vendas_trat["valor_liquido"] > 0,
    vendas_trat["margem_bruta"] / vendas_trat["valor_liquido"],
    np.nan
)

vendas_trat.head()


,id_pedido,data_pedido,id_cliente,id_produto,vendedor,canal,quantidade,preco_unitario,desconto,frete,status_pedido,flag_outlier_preco,valor_bruto,flag_produto_inexistente,flag_cliente_inexistente,valor_liquido,mes,produto,categoria,subcategoria,custo_unitario,custo_total,margem_bruta,margem_%
0,PED00001,2025-08-26,CLI115,PRD021,Não informado,Inside Sales,4.00,908.99,181.80,6.27,Devolvido,False,"3,635.96",False,False,"3,460.43",2025-08-01,Bicicleta Plus,Esporte,Bicicleta,482.22,"1,928.88","1,531.55",0.44
1,PED00002,2025-06-14,CLI157,PRD004,Bruno Costa,Marketplace,2.00,"1,013.19",0.00,18.42,Entregue,False,"2,026.38",False,False,"2,044.80",2025-06-01,Smartwatch Max,Eletrônicos,Smartwatch,727.97,"1,455.94",588.86,0.29
2,PED00003,2025-12-13,CLI114,PRD017,Gustavo Reis,Marketplace,2.00,"1,265.85",126.58,13.82,Cancelado,False,"2,531.70",False,False,"2,418.94",2025-12-01,Creme Facial Plus,Beleza,Creme Facial,759.33,"1,518.66",900.28,0.37
3,PED00004,2025-10-26,CLI022,PRD024,Bruno Costa,Marketplace,2.00,50.68,15.20,54.84,Entregue,False,101.36,False,False,141.00,2025-10-01,Garrafa Térmica Prime,Esporte,Garrafa Térmica,35.58,71.16,69.84,0.50
4,PED00005,2025-03-07,CLI151,PRD023,Diego Rocha,Loja Física,1.00,256.85,12.84,32.86,Entregue,False,256.85,False,False,276.87,2025-03-01,Tênis Corrida Max,Esporte,Tênis Corrida,180.98,180.98,95.89,0.35


In [10]:
vendas

,id_pedido,data_pedido,id_cliente,id_produto,vendedor,canal,quantidade,preco_unitario,desconto,frete,status_pedido
0,PED00001,2025-08-26 00:00:00,CLI115,PRD021,NaN,Inside Sales,4,908.99,181.80,6.27,dev
1,PED00002,2025-06-14 00:00:00,CLI157,PRD004,Bruno Costa,Market Place,2,"1,013.19",0.00,18.42,Entregue
2,PED00003,2025-12-13 00:00:00,CLI114,PRD017,Gustavo Reis,Market Place,2,"1,265.85",126.58,13.82,Cancelado
3,PED00004,2025-10-26 00:00:00,CLI022,PRD024,Bruno Costa,Marketplace,2,50.68,15.20,54.84,ENTREGUE
4,PED00005,2025-03-07 00:00:00,CLI151,PRD023,Diego Rocha,Loja Física,1,256.85,12.84,32.86,ENTREGUE
...,...,...,...,...,...,...,...,...,...,...,...
1295,PED01296,2025-10-05 00:00:00,CLI046,PRD004,Ana Lima,Loja Física,5,"1,029.54",0.00,16.74,Devolvido
1296,PED01297,2025-09-17 00:00:00,CLI028,PRD010,Carla Martins,e-commerce,1,396.65,0.00,23.13,ENTREGUE
1297,PED01298,2025-11-15 00:00:00,CLI006,PRD016,Ana Lima,Loja Física,1,"1,027.44",154.12,62.92,Cancelado
1298,PED01299,2025-03-14 00:00:00,CLI035,PRD017,Fernanda Alves,Loja Fisica,6,"1,247.82",0.00,12.85,Devolvido



## 9. Indicadores finais para validar o tratamento


In [11]:

resumo = pd.DataFrame({
    "indicador": [
        "Pedidos válidos",
        "Receita líquida",
        "Ticket médio",
        "Quantidade vendida",
        "Margem bruta",
        "Margem média %"
    ],
    "valor": [
        vendas_trat["id_pedido"].nunique(),
        vendas_trat["valor_liquido"].sum(),
        vendas_trat["valor_liquido"].sum() / vendas_trat["id_pedido"].nunique(),
        vendas_trat["quantidade"].sum(),
        vendas_trat["margem_bruta"].sum(),
        vendas_trat["margem_bruta"].sum() / vendas_trat["valor_liquido"].sum()
    ]
})

display(resumo)


,indicador,valor
0,Pedidos válidos,"1,290.00"
1,Receita líquida,"2,249,051.17"
2,Ticket médio,"1,743.45"
3,Quantidade vendida,"3,339.00"
4,Margem bruta,"833,441.53"
5,Margem média %,0.37



## 10. Exportação da base tratada

O arquivo gerado pode ser usado diretamente no Power BI.


In [12]:

saida = Path("base_comercial_vendas_tratada.xlsx")

with pd.ExcelWriter(saida, engine="openpyxl") as writer:
    vendas_trat.to_excel(writer, sheet_name="vendas_tratada", index=False)
    produtos_trat.to_excel(writer, sheet_name="produtos_tratada", index=False)
    clientes_trat.to_excel(writer, sheet_name="clientes_tratada", index=False)
    metas_trat.to_excel(writer, sheet_name="metas", index=False)
    resumo.to_excel(writer, sheet_name="resumo_tratamento", index=False)

print(f"Arquivo exportado: {saida.resolve()}")


Arquivo exportado: C:\Users\palom\Downloads\case_comercial_vendas\base_comercial_vendas_tratada.xlsx
